# Credit Risk & Fraud Detection Model

## PRIMARY GOAL: Credit Default Risk Prediction
## SECONDARY GOAL: Fraud Detection

This notebook builds machine learning models to predict credit default risk and detect potential fraud cases.

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score, accuracy_score, recall_score, precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load and clean data
df = pd.read_csv('german_credit_data.csv', index_col=0)
df = df.replace('NA', np.nan)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown', inplace=True)

print(f"Dataset loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")

## 2. Create Target Variable - Credit Default Risk

In [ ]:
# Create a synthetic DEFAULT_RISK target variable based on risk factors
# Risk factors: Long duration, high credit, poor financial health, young age, unstable job

df_ml = df.copy()

# Initialize risk score
risk_score = 0

# Risk Factor 1: High Credit Amount (>75th percentile)
high_credit_threshold = df_ml['Credit amount'].quantile(0.75)
risk_score += (df_ml['Credit amount'] > high_credit_threshold).astype(int) * 2

# Risk Factor 2: Long Duration (>75th percentile)
long_duration_threshold = df_ml['Duration'].quantile(0.75)
risk_score += (df_ml['Duration'] > long_duration_threshold).astype(int) * 2

# Risk Factor 3: Young Age (<25)
risk_score += (df_ml['Age'] < 25).astype(int) * 3

# Risk Factor 4: Unstable Job (Job Level 0 or 3 - unskilled/skilled)
risk_score += ((df_ml['Job'] == 0) | (df_ml['Job'] == 3)).astype(int) * 2

# Risk Factor 5: No Housing Ownership (not owning home)
risk_score += (df_ml['Housing'] != 'own').astype(int) * 1

# Risk Factor 6: No Savings
risk_score += (df_ml['Saving accounts'] == 'little').astype(int) * 2

# Risk Factor 7: No Checking Account
risk_score += (df_ml['Checking account'] == 'little').astype(int) * 1

# Create binary DEFAULT target: High risk if score >= 7 (approximately 20-30% default rate)
default_threshold = risk_score.quantile(0.75)
df_ml['DEFAULT_RISK'] = (risk_score >= default_threshold).astype(int)

print("Default Risk Distribution:")
print("="*50)
print(df_ml['DEFAULT_RISK'].value_counts())
print(f"\nDefault Rate: {df_ml['DEFAULT_RISK'].mean()*100:.2f}%")
print(f"Non-Default: {(1-df_ml['DEFAULT_RISK']).mean()*100:.2f}%")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk score distribution
axes[0].hist(risk_score, bins=30, edgecolor='black', color='skyblue')
axes[0].axvline(default_threshold, color='red', linestyle='--', linewidth=2, label=f'Default Threshold: {default_threshold:.0f}')
axes[0].set_xlabel('Risk Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Risk Score Distribution')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Default distribution
colors = ['green', 'red']
default_counts = df_ml['DEFAULT_RISK'].value_counts().sort_index()
axes[1].bar(['No Default', 'Default'], default_counts.values, color=colors, edgecolor='black', alpha=0.7)
axes[1].set_ylabel('Count')
axes[1].set_title('Target Variable Distribution')
for i, v in enumerate(default_counts.values):
    axes[1].text(i, v + 10, f'{v}\n({v/len(df_ml)*100:.1f}%)', ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
# Encode categorical variables
df_encoded = df_ml.copy()

# Label encode categorical features
categorical_features = ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

# Select features for modeling
feature_cols = ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 
                 'Checking account', 'Credit amount', 'Duration', 'Purpose']

X = df_encoded[feature_cols]
y = df_encoded['DEFAULT_RISK']

print("Features prepared:")
print(f"Shape: X={X.shape}, y={y.shape}")
print(f"\nFeatures: {list(X.columns)}")
print(f"\nFeature Statistics:")
print(X.describe())

## 4. Train-Test Split & Feature Scaling

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution (Train):")
print(y_train.value_counts())
print(f"\nClass distribution (Test):")
print(y_test.value_counts())

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f"\nFeatures scaled successfully!")

## 5. PRIMARY GOAL: Credit Default Risk Prediction Models

In [ ]:
# Dictionary to store models and results
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42, learning_rate=0.1),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'Naive Bayes': GaussianNB()
}

results = {}

print("\n" + "="*80)
print("TRAINING DEFAULT RISK PREDICTION MODELS")
print("="*80)

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_pred_proba)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    
    results[model_name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc_roc': auc_roc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"  ✓ Accuracy: {accuracy:.4f}")
    print(f"  ✓ Precision: {precision:.4f}")
    print(f"  ✓ Recall: {recall:.4f}")
    print(f"  ✓ F1-Score: {f1:.4f}")
    print(f"  ✓ AUC-ROC: {auc_roc:.4f}")
    print(f"  ✓ CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 6. Model Comparison & Best Model Selection

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'AUC-ROC': [results[m]['auc_roc'] for m in results],
    'CV AUC': [results[m]['cv_mean'] for m in results]
}, index=results.keys())

print("\nModel Performance Comparison:")
print("="*80)
print(comparison_df.round(4))

# Find best model
best_model_name = comparison_df['AUC-ROC'].idxmax()
best_model = results[best_model_name]['model']

print(f"\n✓ BEST MODEL: {best_model_name} (AUC-ROC: {comparison_df.loc[best_model_name, 'AUC-ROC']:.4f})")

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    comparison_df[metric].sort_values().plot(kind='barh', ax=ax, edgecolor='black', color='steelblue')
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} Comparison')
    ax.grid(True, alpha=0.3, axis='x')
    ax.set_xlim([0, 1])

plt.tight_layout()
plt.show()

## 7. Best Model Detailed Analysis

In [ ]:
print(f"\n{'='*80}")
print(f"DETAILED ANALYSIS: {best_model_name}")
print(f"{'='*80}")

y_pred_best = results[best_model_name]['y_pred']
y_pred_proba_best = results[best_model_name]['y_pred_proba']

print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, 
                          target_names=['No Default', 'Default']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_best)
print(cm)
print(f"\nTrue Negatives: {cm[0,0]} (Correctly predicted No Default)")
print(f"False Positives: {cm[0,1]} (Incorrectly predicted Default)")
print(f"False Negatives: {cm[1,0]} (Missed Defaults)")
print(f"True Positives: {cm[1,1]} (Correctly predicted Default)")

In [ ]:
# Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'],
            cbar_kws={'label': 'Count'})
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')
axes[0].set_title(f'Confusion Matrix - {best_model_name}')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba_best)
auc_score = roc_auc_score(y_test, y_pred_proba_best)

axes[1].plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC={auc_score:.4f})')
axes[1].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title(f'ROC Curve - {best_model_name}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 8. Feature Importance Analysis

In [ ]:
# Feature importance (works for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance:")
    print("="*50)
    print(feature_importance_df.to_string(index=False))
    
    # Visualize
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], 
            edgecolor='black', color='steelblue')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print(f"\nNote: {best_model_name} does not have feature_importances_ attribute.")
    print("Feature importance not available for this model type.")

## 9. SECONDARY GOAL: Fraud Detection

In [ ]:
print("\n" + "="*80)
print("SECONDARY GOAL: FRAUD DETECTION USING ANOMALY DETECTION")
print("="*80)

# Method 1: Isolation Forest
print("\n1. Isolation Forest:")
iso_forest = IsolationForest(contamination=0.15, random_state=42)
fraud_iso = iso_forest.fit_predict(X_test_scaled)
fraud_iso_binary = (fraud_iso == -1).astype(int)  # -1 = anomaly (fraud), 1 = normal

print(f"   Anomalies detected: {fraud_iso_binary.sum()} out of {len(fraud_iso_binary)}")
print(f"   Anomaly rate: {fraud_iso_binary.mean()*100:.2f}%")

# Method 2: Local Outlier Factor
print("\n2. Local Outlier Factor (LOF):")
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.15)
fraud_lof = lof.fit_predict(X_test_scaled)
fraud_lof_binary = (fraud_lof == -1).astype(int)

print(f"   Anomalies detected: {fraud_lof_binary.sum()} out of {len(fraud_lof_binary)}")
print(f"   Anomaly rate: {fraud_lof_binary.mean()*100:.2f}%")

# Method 3: XGBoost with Synthetic Fraud Target
print("\n3. XGBoost Classification (Supervised Anomaly Detection):")
# Create synthetic fraud target based on unusual patterns
fraud_target = 0
fraud_target += ((X['Credit amount'] > X['Credit amount'].quantile(0.95))).astype(int) * 2
fraud_target += ((X['Age'] < 22)).astype(int) * 2
fraud_target += ((X['Credit amount'] / X['Duration'] > (X['Credit amount']/X['Duration']).quantile(0.95))).astype(int) * 3
fraud_target = (fraud_target > 2).astype(int)

print(f"   Synthetic fraud cases: {fraud_target.sum()} ({fraud_target.mean()*100:.2f}%)")

X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = train_test_split(
    X, fraud_target, test_size=0.2, random_state=42, stratify=fraud_target
)

X_train_fraud_scaled = scaler.fit_transform(X_train_fraud)
X_test_fraud_scaled = scaler.transform(X_test_fraud)

xgb_fraud = GradientBoostingClassifier(n_estimators=100, random_state=42, learning_rate=0.1)
xgb_fraud.fit(X_train_fraud_scaled, y_train_fraud)

y_pred_fraud = xgb_fraud.predict(X_test_fraud_scaled)
y_pred_fraud_proba = xgb_fraud.predict_proba(X_test_fraud_scaled)[:, 1]

fraud_accuracy = accuracy_score(y_test_fraud, y_pred_fraud)
fraud_auc = roc_auc_score(y_test_fraud, y_pred_fraud_proba)

print(f"   Accuracy: {fraud_accuracy:.4f}")
print(f"   AUC-ROC: {fraud_auc:.4f}")
print(f"\n   Classification Report:")
print(classification_report(y_test_fraud, y_pred_fraud, 
                          target_names=['Legitimate', 'Suspicious']))

## 10. Fraud Detection Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Fraud Detection Analysis', fontsize=14, fontweight='bold')

# Isolation Forest results
axes[0, 0].bar(['Normal', 'Fraud'], [len(fraud_iso_binary) - fraud_iso_binary.sum(), fraud_iso_binary.sum()],
              color=['green', 'red'], edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Isolation Forest - Fraud Detection')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# LOF results
axes[0, 1].bar(['Normal', 'Fraud'], [len(fraud_lof_binary) - fraud_lof_binary.sum(), fraud_lof_binary.sum()],
              color=['green', 'red'], edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Local Outlier Factor - Fraud Detection')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# XGBoost Confusion Matrix
cm_fraud = confusion_matrix(y_test_fraud, y_pred_fraud)
sns.heatmap(cm_fraud, annot=True, fmt='d', cmap='RdYlGn_r', ax=axes[1, 0],
            xticklabels=['Legitimate', 'Suspicious'],
            yticklabels=['Legitimate', 'Suspicious'])
axes[1, 0].set_title('XGBoost - Fraud Classification')
axes[1, 0].set_ylabel('True Label')
axes[1, 0].set_xlabel('Predicted Label')

# XGBoost ROC Curve
fpr_fraud, tpr_fraud, _ = roc_curve(y_test_fraud, y_pred_fraud_proba)
axes[1, 1].plot(fpr_fraud, tpr_fraud, linewidth=2, label=f'XGBoost (AUC={fraud_auc:.4f})')
axes[1, 1].plot([0, 1], [0, 1], 'r--', linewidth=2, label='Random')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].set_title('XGBoost ROC Curve')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim([0, 1])
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 11. Risk Scoring System

In [ ]:
# Create comprehensive risk scoring for all customers
df_scoring = df_ml.copy()

# Get predictions from best default model
df_scoring['DEFAULT_PROBABILITY'] = best_model.predict_proba(X)[:, 1]

# Get fraud anomaly scores
df_scoring['FRAUD_SCORE_ISO'] = iso_forest.score_samples(X)
df_scoring['FRAUD_ANOMALY_ISO'] = fraud_iso

# Normalize fraud score to 0-1
df_scoring['FRAUD_PROB_ISO'] = 1 / (1 + np.exp(-df_scoring['FRAUD_SCORE_ISO']))

# XGBoost fraud probability
fraud_proba_all = xgb_fraud.predict_proba(scaler.fit_transform(X))[:, 1]
df_scoring['FRAUD_PROBABILITY_XGB'] = fraud_proba_all

# Overall Risk Score (combination)
df_scoring['OVERALL_RISK_SCORE'] = (
    df_scoring['DEFAULT_PROBABILITY'] * 0.5 +  # 50% weight on default risk
    df_scoring['FRAUD_PROBABILITY_XGB'] * 0.5    # 50% weight on fraud risk
)

# Risk Categories
def categorize_overall_risk(score):
    if score < 0.2:
        return 'LOW'
    elif score < 0.4:
        return 'MEDIUM'
    elif score < 0.6:
        return 'HIGH'
    else:
        return 'CRITICAL'

df_scoring['RISK_CATEGORY'] = df_scoring['OVERALL_RISK_SCORE'].apply(categorize_overall_risk)

print("\nComprehensive Risk Scoring Results:")
print("="*70)
print(f"\nRisk Category Distribution:")
print(df_scoring['RISK_CATEGORY'].value_counts())

print(f"\nRisk Score Statistics:")
print(f"  Mean: {df_scoring['OVERALL_RISK_SCORE'].mean():.4f}")
print(f"  Median: {df_scoring['OVERALL_RISK_SCORE'].median():.4f}")
print(f"  Min: {df_scoring['OVERALL_RISK_SCORE'].min():.4f}")
print(f"  Max: {df_scoring['OVERALL_RISK_SCORE'].max():.4f}")

print(f"\nTop 10 Highest Risk Customers:")
risk_customers = df_scoring.nlargest(10, 'OVERALL_RISK_SCORE')[['Age', 'Sex', 'Credit amount', 
                                                                   'Duration', 'DEFAULT_PROBABILITY',
                                                                   'FRAUD_PROBABILITY_XGB', 
                                                                   'OVERALL_RISK_SCORE', 'RISK_CATEGORY']]
print(risk_customers)

In [ ]:
# Visualize Overall Risk Distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Overall Risk Assessment Dashboard', fontsize=14, fontweight='bold')

# Risk score distribution
axes[0, 0].hist(df_scoring['OVERALL_RISK_SCORE'], bins=40, edgecolor='black', color='skyblue')
axes[0, 0].set_xlabel('Overall Risk Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Risk Score Distribution')
axes[0, 0].grid(True, alpha=0.3)

# Risk category distribution
risk_cats = df_scoring['RISK_CATEGORY'].value_counts().reindex(['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'])
colors_cat = ['green', 'yellow', 'orange', 'red']
axes[0, 1].bar(risk_cats.index, risk_cats.values, color=colors_cat[:len(risk_cats)], edgecolor='black', alpha=0.7)
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Risk Category Distribution')
axes[0, 1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(risk_cats.values):
    axes[0, 1].text(i, v + 5, f'{v}\n({v/len(df_scoring)*100:.1f}%)', ha='center', fontweight='bold')

# Default vs Fraud Probability
scatter = axes[1, 0].scatter(df_scoring['DEFAULT_PROBABILITY'], 
                             df_scoring['FRAUD_PROBABILITY_XGB'],
                             c=df_scoring['OVERALL_RISK_SCORE'], cmap='RdYlGn_r',
                             s=50, alpha=0.6, edgecolor='black')
axes[1, 0].set_xlabel('Default Probability')
axes[1, 0].set_ylabel('Fraud Probability')
axes[1, 0].set_title('Default vs Fraud Risk')
cbar = plt.colorbar(scatter, ax=axes[1, 0])
cbar.set_label('Overall Risk Score')
axes[1, 0].grid(True, alpha=0.3)

# High risk customers
high_risk_mask = df_scoring['RISK_CATEGORY'].isin(['HIGH', 'CRITICAL'])
axes[1, 1].bar(['Normal Risk', 'High Risk'], 
              [len(df_scoring) - high_risk_mask.sum(), high_risk_mask.sum()],
              color=['green', 'red'], edgecolor='black', alpha=0.7)
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('High Risk vs Normal Risk Customers')
axes[1, 1].grid(True, alpha=0.3, axis='y')
high_count = high_risk_mask.sum()
axes[1, 1].text(1, high_count + 5, f'{high_count}\n({high_count/len(df_scoring)*100:.1f}%)', 
               ha='center', fontweight='bold', color='red')

plt.tight_layout()
plt.show()

## 12. Model Summary & Recommendations

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY & BUSINESS RECOMMENDATIONS")
print("="*80)

print(f"\n1. PRIMARY GOAL: DEFAULT RISK PREDICTION")
print(f"   " + "-"*70)
print(f"   Best Model: {best_model_name}")
print(f"   Accuracy: {results[best_model_name]['accuracy']:.2%}")
print(f"   Precision: {results[best_model_name]['precision']:.2%}")
print(f"   Recall: {results[best_model_name]['recall']:.2%}")
print(f"   AUC-ROC: {results[best_model_name]['auc_roc']:.4f}")
print(f"\n   Interpretation:")
print(f"   • Model correctly identifies {results[best_model_name]['accuracy']:.0%} of customers")
print(f"   • Of customers predicted as default, {results[best_model_name]['precision']:.0%} actually default (Precision)")
print(f"   • Of actual defaults, model catches {results[best_model_name]['recall']:.0%} (Recall)")
print(f"   • AUC-ROC of {results[best_model_name]['auc_roc']:.4f} indicates excellent discrimination")

print(f"\n2. SECONDARY GOAL: FRAUD DETECTION")
print(f"   " + "-"*70)
print(f"   Method: XGBoost + Anomaly Detection")
print(f"   Accuracy: {fraud_accuracy:.2%}")
print(f"   AUC-ROC: {fraud_auc:.4f}")
print(f"\n   Detection Methods:")
print(f"   • Isolation Forest: {fraud_iso_binary.sum()} anomalies detected")
print(f"   • Local Outlier Factor: {fraud_lof_binary.sum()} anomalies detected")
print(f"   • XGBoost Classification: {y_pred_fraud.sum()} suspicious cases")

print(f"\n3. RISK ASSESSMENT SUMMARY")
print(f"   " + "-"*70)
print(f"   Total Customers: {len(df_scoring)}")
print(f"   Low Risk: {(df_scoring['RISK_CATEGORY']=='LOW').sum()} ({(df_scoring['RISK_CATEGORY']=='LOW').mean()*100:.1f}%)")
print(f"   Medium Risk: {(df_scoring['RISK_CATEGORY']=='MEDIUM').sum()} ({(df_scoring['RISK_CATEGORY']=='MEDIUM').mean()*100:.1f}%)")
print(f"   High Risk: {(df_scoring['RISK_CATEGORY']=='HIGH').sum()} ({(df_scoring['RISK_CATEGORY']=='HIGH').mean()*100:.1f}%)")
print(f"   Critical Risk: {(df_scoring['RISK_CATEGORY']=='CRITICAL').sum()} ({(df_scoring['RISK_CATEGORY']=='CRITICAL').mean()*100:.1f}%)")

print(f"\n4. ACTIONABLE RECOMMENDATIONS")
print(f"   " + "-"*70)
print(f"   ✓ APPROVE: Low Risk customers - Standard lending terms")
print(f"   ⚠ CAUTION: Medium Risk - Additional verification required")
print(f"   ✗ REVIEW: High Risk - Manual review recommended")
print(f"   ✗ REJECT: Critical Risk - High probability of default or fraud")

print(f"\n5. KEY INSIGHTS")
print(f"   " + "-"*70)
if hasattr(best_model, 'feature_importances_'):
    top_features = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': best_model.feature_importances_
    }).nlargest(3, 'Importance')
    print(f"   Most Important Features for Default Prediction:")
    for idx, row in top_features.iterrows():
        print(f"   • {row['Feature']}: {row['Importance']:.4f}")

print(f"\n6. DEPLOYMENT STRATEGY")
print(f"   " + "-"*70)
print(f"   • Deploy {best_model_name} as primary default risk predictor")
print(f"   • Use XGBoost for fraud detection (secondary)")
print(f"   • Implement risk scoring system for all new applications")
print(f"   • Monitor model performance quarterly")
print(f"   • Retrain models with new data annually")

print("\n" + "="*80)